In [1]:
from pathlib import Path
import subprocess
import json
import time
from datetime import datetime
from collections import Counter

In [ ]:
# Notebook location:
# MSC_RESEARCH_PROJECT/progress/markdown_extraction/markdown_extraction.ipynb

PROJECT_ROOT = Path.cwd().parents[1]

PDF_SENTIMENT_DIR = PROJECT_ROOT / "pdfs" / "sentiment" 
MARKDOWN_SENTIMENT_DIR = PROJECT_ROOT / "markdown" / "sentiment"
LOG_DIR = PROJECT_ROOT / "progress" / "markdown_extraction" / "logs"

MARKDOWN_SENTIMENT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Sentiment PDF dir:", PDF_SENTIMENT_DIR)
print("Sentiment Markdown dir:", MARKDOWN_SENTIMENT_DIR)

Project root: /home/nsirim/Jupyter/myvenv/home/mariem/tu_dublin/msc_research_project
Sentiment PDF dir: /home/nsirim/Jupyter/myvenv/home/mariem/tu_dublin/msc_research_project/pdfs/sentiment/eu_all
Sentiment Markdown dir: /home/nsirim/Jupyter/myvenv/home/mariem/tu_dublin/msc_research_project/markdown/sentiment


In [3]:
def check_docling():
    result = subprocess.run(
        ["docling", "--version"],
        capture_output=True,
        text=True
    )
    
    if result.returncode != 0:
        raise RuntimeError(
            "Docling is not available in this environment. "
            "Activate the correct virtual environment or install it with: pip install docling"
        )
    
    print("Docling is available:")
    print(result.stdout.strip() or result.stderr.strip())

check_docling()

Docling is available:
Docling version: 2.95.0
Docling Core version: 2.77.1
Docling IBM Models version: 3.13.2
Docling Parse version: 5.11.0
Python: cpython-313 (3.13.5)
Platform: Linux-6.12.90+deb13-amd64-x86_64-with-glibc2.41


In [4]:
def find_sentiment_pdfs():
    if not PDF_SENTIMENT_DIR.exists():
        raise FileNotFoundError(f"Missing folder: {PDF_SENTIMENT_DIR}")
    
    pdf_files = sorted(PDF_SENTIMENT_DIR.glob("*.pdf"))
    
    records = []
    for pdf_path in pdf_files:
        records.append({
            "corpus_layer": "sentiment",
            "country": "EU",
            "pdf_path": pdf_path,
            "output_dir": MARKDOWN_SENTIMENT_DIR
        })
    
    return records


sentiment_pdf_records = find_sentiment_pdfs()

print(f"Found {len(sentiment_pdf_records)} sentiment PDF files.")
for item in sentiment_pdf_records[:20]:
    print(item["pdf_path"].name)

Found 18 sentiment PDF files.
AI competency framework  for students.pdf
AI is a threat to the essence of education – The Irish Times.pdf
How are teachers in Europe using AI in the classroom and which country uses it the most_ _ Euronews.pdf
How do the French relate to artificial intelligence_ - Labo.pdf
IMPORTANT_A_051024_Insights into AI and the youth sector.pdf
IMPORTANT_A_generative artificial intelligence in secondary education-KJ0125604ENN.pdf
IMPORTANT_B_AI_views of youth workers.pdf
IMPORTANT_B_EPRS_ATA(2025)769494_EN.pdf
IMPORTANT_B_artificial intelligence in classrooms-QA0126065ENN.pdf
IMPORTANT_EU_Survey on artificial intelligence for teaching and learning – Results _ European School Education Platform.pdf
IMPORTANT_FRANCE_20251119-Les-Francais-lIA.pdf
IMPORTANT_FRANCE_AI4T_WP3_D3.3_NR_Ireland.pdf
IMPORTANT_FRANCE_C_French digital society_ digital mediation professionals face the challenge of digital remoteness - Labo.pdf
IMPORTANT_FRANCE_Les jeunes Français face à l’IA génér

In [5]:
def convert_pdf_to_markdown(
    pdf_path: Path,
    output_dir: Path,
    *,
    overwrite: bool = False,
    enrich_pictures: bool = False,
    enrich_charts: bool = False,
    device: str = "cpu",
    timeout_seconds: int = 900
):
    """
    Convert a single PDF to Markdown using Docling.
    """

    output_dir.mkdir(parents=True, exist_ok=True)
    expected_md = output_dir / f"{pdf_path.stem}.md"

    if expected_md.exists() and not overwrite:
        return {
            "pdf": str(pdf_path),
            "output_dir": str(output_dir),
            "status": "skipped_exists",
            "markdown": str(expected_md),
            "returncode": 0,
            "stdout": "",
            "stderr": ""
        }

    cmd = [
        "docling",
        str(pdf_path),
        "--from", "pdf",
        "--to", "md",
        "--output", str(output_dir),
        "--image-export-mode", "referenced",
        "--ocr",
        "--tables",
        "--table-mode", "accurate",
        "--device", device,
        "--document-timeout", str(timeout_seconds),
        "--num-threads", "4",
    ]

    if enrich_pictures:
        cmd.append("--enrich-picture-description")

    if enrich_charts:
        cmd.append("--enrich-chart-extraction")

    started = time.time()

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    elapsed = round(time.time() - started, 2)
    status = "success" if result.returncode == 0 else "failed"

    return {
        "pdf": str(pdf_path),
        "output_dir": str(output_dir),
        "status": status,
        "markdown": str(expected_md) if expected_md.exists() else None,
        "returncode": result.returncode,
        "elapsed_seconds": elapsed,
        "stdout": result.stdout,
        "stderr": result.stderr,
        "command": " ".join(cmd)
    }

In [6]:
sentiment_conversion_logs = []

for index, record in enumerate(sentiment_pdf_records, start=1):
    pdf_path = record["pdf_path"]
    output_dir = record["output_dir"]

    print(f"[{index}/{len(sentiment_pdf_records)}] Converting sentiment PDF: {pdf_path.name}")

    log = convert_pdf_to_markdown(
        pdf_path=pdf_path,
        output_dir=output_dir,
        overwrite=False,
        enrich_pictures=False,
        enrich_charts=False,
        device="cpu",
        timeout_seconds=900
    )

    log["corpus_layer"] = "sentiment"
    log["country"] = "EU"

    sentiment_conversion_logs.append(log)

    print("  status:", log["status"])
    if log["status"] == "failed":
        print("  stderr:", log["stderr"][:1200])

[1/18] Converting sentiment PDF: AI competency framework  for students.pdf
  status: success
[2/18] Converting sentiment PDF: AI is a threat to the essence of education – The Irish Times.pdf
  status: success
[3/18] Converting sentiment PDF: How are teachers in Europe using AI in the classroom and which country uses it the most_ _ Euronews.pdf
  status: success
[4/18] Converting sentiment PDF: How do the French relate to artificial intelligence_ - Labo.pdf
  status: success
[5/18] Converting sentiment PDF: IMPORTANT_A_051024_Insights into AI and the youth sector.pdf
  status: success
[6/18] Converting sentiment PDF: IMPORTANT_A_generative artificial intelligence in secondary education-KJ0125604ENN.pdf
  status: success
[7/18] Converting sentiment PDF: IMPORTANT_B_AI_views of youth workers.pdf
  status: success
[8/18] Converting sentiment PDF: IMPORTANT_B_EPRS_ATA(2025)769494_EN.pdf
  status: success
[9/18] Converting sentiment PDF: IMPORTANT_B_artificial intelligence in classrooms-QA01

In [7]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = LOG_DIR / f"docling_sentiment_conversion_{timestamp}.json"

with open(log_file, "w", encoding="utf-8") as f:
    json.dump(sentiment_conversion_logs, f, indent=2, ensure_ascii=False)

print("Saved sentiment conversion log to:", log_file)

Saved sentiment conversion log to: /home/nsirim/Jupyter/myvenv/home/mariem/tu_dublin/msc_research_project/progress/text_extraction/logs/docling_sentiment_conversion_20260528_011028.json


In [8]:
status_counts = Counter(item["status"] for item in sentiment_conversion_logs)

print("Sentiment conversion summary:")
for status, count in status_counts.items():
    print(f"  {status}: {count}")

failed = [item for item in sentiment_conversion_logs if item["status"] == "failed"]

if failed:
    print("\nFailed files:")
    for item in failed:
        print("-", item["pdf"])

Sentiment conversion summary:
  success: 18


In [9]:
generated_sentiment_markdowns = sorted(MARKDOWN_SENTIMENT_DIR.glob("*.md"))

print(f"Generated sentiment Markdown files: {len(generated_sentiment_markdowns)}")

for md_path in generated_sentiment_markdowns[:30]:
    print(md_path.relative_to(PROJECT_ROOT))

Generated sentiment Markdown files: 18
markdown/sentiment/AI competency framework  for students.md
markdown/sentiment/AI is a threat to the essence of education – The Irish Times.md
markdown/sentiment/How are teachers in Europe using AI in the classroom and which country uses it the most_ _ Euronews.md
markdown/sentiment/How do the French relate to artificial intelligence_ - Labo.md
markdown/sentiment/IMPORTANT_A_051024_Insights into AI and the youth sector.md
markdown/sentiment/IMPORTANT_A_generative artificial intelligence in secondary education-KJ0125604ENN.md
markdown/sentiment/IMPORTANT_B_AI_views of youth workers.md
markdown/sentiment/IMPORTANT_B_EPRS_ATA(2025)769494_EN.md
markdown/sentiment/IMPORTANT_B_artificial intelligence in classrooms-QA0126065ENN.md
markdown/sentiment/IMPORTANT_EU_Survey on artificial intelligence for teaching and learning – Results _ European School Education Platform.md
markdown/sentiment/IMPORTANT_FRANCE_20251119-Les-Francais-lIA.md
markdown/sentiment/I